# NYC Taxi Dataset - Big Data e Computação em Nuvem

Profs. Thanuci & Michel

**Do que praticamos na aula passada para o que vamos praticar hoje: Spark Data Frames**

Tabela comparativa entre a programação utilizando Dask, Spark RDDs e Spark Data Frames.


| Característica           | Dask                                   | Spark RDDs                    | Spark DataFrames                                 |
|--------------------------|----------------------------------------|--------------------------------------|--------------------------------------------------------|
| **Modelo de Programação**| APIs paralelas para arrays, dataframes e listas. Agendamento dinâmico e execução em tempo real. | Baseado em RDDs, coleções distribuídas de objetos imutáveis. Opera em lotes. | APIs de alto nível para consultas e manipulação de dados estruturados, inspirado no pandas. |
| **Escala e Performance** | Escala de uma máquina a clusters; útil mesmo em hardware limitado. Bom para dados além da memória. | Ideal para grandes volumes de dados em clusters. Alta eficiência com processamento em memória. | Otimizado para grandes volumes de dados com Catalyst optimizer e Tungsten execution engine. |
| **Facilidade de Uso**    | Fácil integração com NumPy, pandas e scikit-learn. Favorável para usuários desses pacotes. | Curva de aprendizado mais íngreme. Integração com SQL, machine learning, processamento de grafos e streaming. | Mais fácil de usar que RDDs, com API semelhante ao pandas e suporte completo a SQL. |
| **Comunidade e Suporte** | Comunidade menor, **mas crescente**. Popular na comunidade de ciência de dados Python. | Amplo suporte de uma grande comunidade e uso industrial. Suporte comercial extenso. | Forte suporte da comunidade, amplamente utilizado na indústria, com recursos extensivos de suporte comercial. |
| **Caso de Uso**          | Análises interativas em larga escala, integração com ferramentas de análise de dados Python. | Processamento de dados em grande escala, transformações complexas, alta tolerância a falhas. | Análise de dados estruturados, consultas SQL, integração com BI e ferramentas de análise avançada. |
| **Tolerância a Falhas**  | Requer configurações específicas para tolerância a falhas; mais limitada em comparação ao Spark. | Alta tolerância a falhas com recomputação automática de partições perdidas. | Alta tolerância a falhas, com recuperação automática de dados perdidos, semelhante aos RDDs. |
| **Execução de Tarefas**  | Agendamento dinâmico, execução em tempo real, bom para pipelines interativos e desenvolvimento incremental. | Execução em lotes com planejamento de tarefas avançado para otimização de performance. | Execução otimizada com Catalyst optimizer, ideal para operações de ETL e análises de dados complexas. |
| **Ecosistema**           | Integra-se bem com outras ferramentas do ecossistema Python como Jupyter Notebooks. | Rico ecossistema com suporte a SQL (Spark SQL), machine learning (MLlib), processamento de grafos (GraphX) e streaming (Spark Streaming). | Integração perfeita com Spark SQL, MLlib, GraphX e Spark Streaming, facilitando análise de dados estruturados e não estruturados. |
| **Tipo de Dados**        | Bem adequado para dados tabulares (DataFrames) e matrizes (Arrays), além de listas de tarefas. | Suporte robusto para dados estruturados e não estruturados, adequado para transformações complexas e análises de dados distribuídas. | Ideal para dados estruturados, com suporte completo a tipos de dados complexos e esquemas, e integração com fontes de dados variadas. |


Neste laboratório vamos explorar o **dataset de corridas realizadas com os Táxis Amarelos de Nova York**.

As colunas que vamos trabalhar são:

* **vendor_id**: A code indicating the TPEP provider that provided the record. 1 = Creative Mobile Technologies, LLC; 2 = VeriFone Inc
* **pickup_datetime**: The date and time when the meter was engaged.
* **dropoff_datetime**: The date and time when the meter was disengaged
* **passenger_count**: The number of passengers in the vehicle. This is a driver-entered value
* **trip_distance**: The elapsed trip distance in miles reported by the taximeter.
* **rate_code**: The final rate code in effect at the end of the trip. 1 = Standard rate 2 = JFK 3 = Newark 4 = Nassau or Westchester 5 = Negotiated fare 6 = Group ride
* **store_and_fwd_flag**: This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server. Y = store and forward trip N = not a store and forward trip
* **payment_type**: A numeric code signifying how the passenger paid for the trip. 1 = Credit card 2 = Cash 3 = No charge 4 = Dispute 5 = Unknown 6 = Voided trip
* **fare_amount**: The time-and-distance fare calculated by the meter
* **extra Miscellaneous extras and surcharges**: Currently, this only includes the \$0.50 and 1 rush hour and overnight charges.
* **mta_tax**: \$0.50 MTA tax that is automatically triggered based on the metered rate in use
* **tip_amount**: Tip amount – This field is automatically populated for credit card tips. Cash tips are not included
* **tolls_amount**: Total amount of all tolls paid in trip.
* **imp_surcharge**: \$0.30 improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015.
* **total_amount**: The total amount charged to passengers. Does not include cash tips
* **pickup_location_id**: TLC Taxi Zone in which the taximeter was engaged
* **dropoff_location_id**: TLC Taxi Zone in which the taximeter was disengaged

## Coleta dos dados - NYC Open Data

[NYC Open Data](https://opendata.cityofnewyork.us/)

NYC Open Data torna a riqueza de dados públicos gerados por várias agências da cidade de Nova York e outras organizações da cidade disponíveis para uso público

## Import de bibliotecas

In [ ]:
import seaborn as sns
import pyspark.sql.functions as f
from pyspark.sql.types import StringType
from matplotlib import pyplot as plt

## Criação Spark Session

In [ ]:
# Criar a sessao do Spark
from pyspark.sql import SparkSession
spark = SparkSession \
                    .builder \
                    .master('local[*]') \
                    .appName('nyctaxi_Thanuci') \
                    .getOrCreate()

In [ ]:
spark

Observem que aqui não carregamos mais o ```spark.context```, pois não estamos trabalhando com RDDs.

## **Seção para consulta**

### Algumas limpezas

Para facilidade, realizamos a extração do dataset e adicionamos à pasta `/FileStore/tables/downloaded_files_taxi/` para que todos possam acessar:

``` python
df = spark.read \
    .option('inferSchema', True) \
    .option('header', True) \
    .csv("/FileStore/tables/downloaded_files_taxi/*.csv")
```

* Contando o número de linhas da base:

``` python 
df.count()
````

* Utilizando apenas uma fração de 10% da base original:

```python
df = df.sample(fraction=0.1, seed=42)
````

* Primeiro fizemos a conversão para double e depois para timestamp. Isso, porque é necessário o aproveitamento de qualquer fração de segundos embutida no dado:

```python
df = df.withColumn('pickup_datetime', f.unix_timestamp('tpep_pickup_datetime', 'MM/dd/yyyy hh:mm:ss a').cast("double").cast("timestamp"))
df = df.withColumn('dropoff_datetime', f.unix_timestamp('tpep_dropoff_datetime', 'MM/dd/yyyy hh:mm:ss a').cast("double").cast("timestamp"))
````

* Uma maneira elegante de dropar colunas irrelevantes:

```python
drop_cols = 'tpep_pickup_datetime','tpep_dropoff_datetime', 'PULocationID', 'DOLocationID'

df = df.drop(*drop_cols)
````

* Contando novamente a base:

```python
df.count()
````

* Salvando o sample em um arquivo separado:

```python
df.write.option("header",True).csv("../10_dados/nyc_taxi/nyc-taxi-2018-sample")
```

## Leitura dos Dados

```python
my_schema = StructType([StructField('ref_coluna', StringType(), True)])
```

In [ ]:
from pyspark.sql.types import *

labels = (('key', TimestampType()),
          ('fare_amount', FloatType()),
          ('pickup_datetime', TimestampType()),
          ('pickup_longitude', FloatType()),
          ('pickup_latitude', FloatType()),
          ('dropoff_longitude', FloatType()),
          ('dropoff_latitude', FloatType()),
          ('passenger_count', IntegerType()))
          

schema = StructType([StructField(x[0], x[1], True) for x in labels])

In [ ]:
df = spark.read.csv("train.csv", header=True, schema=schema)

In [ ]:
df.columns

In [ ]:
df = df.sample(withReplacement=False, fraction=0.01, seed=42)

## Data Quality

* [pyspark.sql.DataFrame.select](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html)
* [pyspark.sql.DataFrame.describe](https://spark.apache.org/docs/3.3.0/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.describe.html)
* [pyspark.sql.DataFrame.withColumn](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)
* [pyspark.sql.functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
    * `date_trunc`
    * `hour`
    * `unix_timestamp`
    * `mean`
    * `stddev`

É necessário realizarmos algumas análises de qualidades de dados, para descobrirmos se este dataset possui problemas associados à coleta das informações.

In [ ]:
df.limit(10).toPandas()

In [ ]:
df.schema

**Observação**: o comando abaixo tira um dataframe / rdd do cache. Como explicado em sala, seria mais rápido tirar o `df` do cache após cachear o `df2`, pois otimiza o tempo de cálculo do `df2`. Mas a preocupação aqui foi liberar memória RAM antes de cachear mais coisas. Como disse, a decisão do que cachear ou não depende do programador, e é uma boa prática para acelerar os códigos!

# Análise Exploratória

## Frequência de corridas diárias

## Variação da frequência de corridas ao longo do dia

In [ ]:
df_hours_trips = df2.withColumn("month", f.date_trunc("month", df2.pickup_datetime))

In [ ]:
df_hours_trips = df_hours_trips.withColumn("hour", f.hour(df_hours_trips.pickup_datetime))

In [ ]:
df_hours_trips = df_hours_trips.groupBy(["month", "hour"]).count().toPandas()

In [ ]:
plt.figure(figsize=(15,5))
ax = sns.lineplot(data=df_hours_trips, x="hour", y="count")
ax.set(xticks=df_hours_trips.hour.unique());

In [ ]:
df_stats.toPandas()

# Pratique

1. Realize mais análises nas colunas, que resulte em uma visualização
2. Notou números estranhos na análise? Realize mais limpezas no dataset.
3. Plote a correlação das variáveis numéricas. Qual é a coluna que mais se correlaciona com a coluna `fare_amount`?